In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU VRAM (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )

assert sys.version_info >= (3, 12), "LeRobot v0.6.1 要求 Python >= 3.12"
assert torch.cuda.is_available(), "请先启用 Colab GPU 运行时"


In [ ]:
%cd /content

!git clone --depth 1 --branch v0.6.1 \
    https://github.com/huggingface/lerobot.git \
    /content/lerobot

In [ ]:
%pip install -e "/content/lerobot[smolvla,libero,evaluation]"

In [ ]:
import torch
import lerobot
import libero
import mujoco
import shutil

from importlib.metadata import version

print("LeRobot:", version("lerobot"))
print("LeRobot path:", lerobot.__file__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("LIBERO:", libero.__file__)
print("MuJoCo:", mujoco.__version__)
print("lerobot-eval:", shutil.which("lerobot-eval"))

In [ ]:
%cd /content

!git clone https://github.com/30thinkdiff/robot-lab.git

%cd /content/robot-lab

!ls -lh studies/policy_generalization/
!git log -1 --oneline

实验A

In [ ]:
!RUN_NAME=baseline_pilot_001 \
N_EPISODES=1 \
EVAL_DRIVER=native \
OUTPUT_ROOT=/content/outputs \
bash scripts/eval_libero.sh

In [ ]:
!find /content/outputs/baseline_pilot_001 \
    -maxdepth 4 -type f

In [ ]:
import json

path = "/content/outputs/baseline_pilot_001/eval_info.json"

with open(path) as f:
    results = json.load(f)

print("Overall:", results.get("overall"))
print("Per-task:", results.get("per_task"))

实验B

In [ ]:
!RUN_NAME=brightness_identity_001 \
N_EPISODES=1 \
EVAL_DRIVER=brightness \
BRIGHTNESS_FACTOR=1.0 \
DEBUG_DIR=/content/outputs/debug_identity \
OUTPUT_ROOT=/content/outputs \
bash scripts/eval_libero.sh

实验C

In [ ]:
!RUN_NAME=brightness_060_pilot_001 \
N_EPISODES=1 \
EVAL_DRIVER=brightness \
BRIGHTNESS_FACTOR=0.6 \
DEBUG_DIR=/content/outputs/debug_brightness_060 \
OUTPUT_ROOT=/content/outputs \
bash scripts/eval_libero.sh

In [ ]:
!find /content/outputs/debug_brightness_060 \
    -type f

In [ ]:
from IPython.display import display, Image
from pathlib import Path

debug_dir = Path("/content/outputs/debug_brightness_060")

for path in sorted(debug_dir.glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/outputs")
dst = Path("/content/drive/MyDrive/robot-lab/experiments")

dst.mkdir(parents=True, exist_ok=True)

for run_name in [
    "baseline_pilot_001",
    "brightness_identity_001",
    "brightness_060_pilot_001",
]:
    run_src = src / run_name

    if run_src.exists():
        shutil.copytree(
            run_src,
            dst / run_name,
            dirs_exist_ok=True
        )

print("已复制现有实验结果到:", dst)

In [ ]:
from pathlib import Path
from IPython.display import Video, display

root = Path("/content/drive/MyDrive/robot-lab/experiments")

videos = list(
    (root / "baseline_pilot_001").rglob("eval_episode_0.mp4")
)

for video in videos:
    print("Video:", video)
    print("Size (MB):", round(video.stat().st_size / 1024**2, 2))
    display(Video(filename=str(video), embed=True))